# TTS Dataset Publish

Reads an identity manifest (scored, per [`docs/manifest.md`](../docs/manifest.md))
and, optionally, a selection file, and builds a Hugging Face dataset folder:
a `wavs/` directory, an LJSpeech-style `metadata.csv`, the full `manifest.jsonl`
with a `selected` flag on every row, and a generated `README.md` dataset card.
It then optionally pushes the folder to the Hugging Face Hub.

This is step 9, Publish, of the [TTS dataset pipeline](../docs/pipeline.md). It
runs alone: give it a manifest path and it does not need any other notebook to
have run in the same session. When no manifest is given it builds one from a
few real `edge-tts` clips, so the notebook has something to publish on its own.

**Inputs**
- `MANIFEST_PATH`: a JSON Lines manifest, format in [`docs/manifest.md`](../docs/manifest.md).
- `SELECTION_PATH`: optional. A JSON Lines file of kept `id` values, format in
  the same document. Absent, every row in the manifest is published.

**Outputs**
- `OUTPUT_DIR/wavs/<id>.wav`: one WAV per selected row, hardlinked from the
  manifest's audio when on the same filesystem, copied otherwise.
- `OUTPUT_DIR/metadata.csv`: `<id>|<text>`, no header.
- `OUTPUT_DIR/manifest.jsonl`: every row of the input manifest, selected or
  not, each with a `selected` boolean, and its `.meta.json` sidecar.
- `OUTPUT_DIR/README.md`: the dataset card, generated from the manifest and
  its sidecars.
- A push to the Hugging Face Hub, only when `HF_PUSH=1`.

**Hardware.** CPU only. No GPU is used or needed. The only network calls are
the optional Hub push and, in the demo path, three short `edge-tts` requests.


---
## NGI0 Grant Attribution

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.
> European Commission Next Generation Internet programme — DG CONNECT,
> grant agreement No 101135429.
> Additional funding: Swiss State Secretariat for Education, Research and Innovation (SERI).

**NGI work item:** WP3 — TTS dataset publish
**CI execution:** runs on every push; the built-in demo needs no GPU and no large download.

---


---
## 0 · Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before**
running the config cell.


In [1]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


DEMO_DIR = _default_base_dir("tts_publish_demo")

# ──────────────────────────────────────────────
# INPUT  (identity manifest and optional selection, docs/manifest.md)
#   Leave MANIFEST_PATH at its default and it is built from three real
#   edge-tts clips, so the notebook has something to publish on its own.
#   Set it to a real manifest to publish real data.
# ──────────────────────────────────────────────
os.environ.setdefault("MANIFEST_PATH",   os.path.join(DEMO_DIR, "src_manifest", "manifest.jsonl"))
os.environ.setdefault("SELECTION_PATH",  "")   # JSON Lines of kept ids; empty = every row is published

# ──────────────────────────────────────────────
# OUTPUT  (the Hugging Face dataset folder)
# ──────────────────────────────────────────────
os.environ.setdefault("OUTPUT_DIR",      os.path.join(DEMO_DIR, "dataset"))

# ──────────────────────────────────────────────
# HUGGING FACE PUSH  (off by default)
# ──────────────────────────────────────────────
os.environ.setdefault("HF_PUSH",         "0")   # 1 = push OUTPUT_DIR to the Hub
os.environ.setdefault("HF_REPO",         "")    # e.g. TigreGotico/tts-train-synthetic-miro_pt-PT
os.environ.setdefault("HF_TOKEN",        "")    # write token (or use huggingface-cli login)


def _env(key, default=""):
    return os.environ.get(key, default)


def _flag(key):
    return _env(key, "0").strip().lower() in ("1", "true", "yes")


print("Configuration loaded.")
print(f"  Manifest    : {_env('MANIFEST_PATH')}")
print(f"  Selection   : {_env('SELECTION_PATH') or '(none — every row is published)'}")
print(f"  Output dir  : {_env('OUTPUT_DIR')}")
print(f"  HF push     : {'yes -> ' + _env('HF_REPO') if _flag('HF_PUSH') else 'no'}")


Configuration loaded.
  Manifest    : ./tts_publish_demo/src_manifest/manifest.jsonl
  Selection   : (none — every row is published)
  Output dir  : ./tts_publish_demo/dataset
  HF push     : no


---
## 1 · Manifest and selection

When `MANIFEST_PATH` does not exist, this stage builds one: three short
`edge-tts` clips with made-up scores, and a fourth clip whose made-up
character error rate (`cer`) is deliberately over a `0.10` bar, together with
a selection file that keeps the first three and drops the fourth. This is the
notebook's self-contained demo path — it needs no manifest to be handed to it,
only outbound access to the `edge-tts` service.

| Env var | Meaning |
|---|---|
| `MANIFEST_PATH` | The identity manifest to publish from. |
| `SELECTION_PATH` | Optional selection file. Empty means every row is published. |


In [2]:
demo_needed = not os.path.exists(_env("MANIFEST_PATH"))

if demo_needed:
    # edge-tts needs outbound internet access; only pulled in for the demo.
    !pip install --quiet --break-system-packages --pre "edge-tts>=7.2.8" "nest_asyncio>=1.6.0" "soundfile>=0.14.0"
else:
    print("MANIFEST_PATH exists — the built-in demo is skipped.")


In [3]:
import asyncio
import json
import shutil
import subprocess
from pathlib import Path

if demo_needed:
    import nest_asyncio
    nest_asyncio.apply()

    manifest_path = Path(_env("MANIFEST_PATH"))
    src_dir = manifest_path.parent
    if src_dir.exists():
        shutil.rmtree(src_dir)
    src_dir.mkdir(parents=True)

    # One clip per call, never padded (see docs/pipeline.md): each row is its
    # own edge-tts request and its own ffmpeg conversion.
    DEMO_CLIPS = [
        {"id": "clip_0001", "text": "Hey Mycroft, what is the weather like today in Lisbon?", "cer": 0.02, "utmos": 3.8},
        {"id": "clip_0002", "text": "Set a timer for ten minutes.", "cer": 0.05, "utmos": 3.6},
        {"id": "clip_0003", "text": "Play some jazz music in the kitchen.", "cer": 0.01, "utmos": 3.9},
        # Made to FAIL the selection rule below: a cer over the 0.10 bar.
        {"id": "clip_0004", "text": "Turn off the lights in the living room.", "cer": 0.34, "utmos": 3.7},
    ]
    DEMO_VOICE = "en-US-JennyNeural"

    async def _synth(text, voice, out_mp3):
        import edge_tts
        await edge_tts.Communicate(text, voice).save(str(out_mp3))

    import soundfile as sf

    demo_rows = []
    for clip in DEMO_CLIPS:
        mp3_path = src_dir / f"{clip['id']}.mp3"
        wav_path = src_dir / f"{clip['id']}.wav"
        asyncio.run(_synth(clip["text"], DEMO_VOICE, mp3_path))
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp3_path),
             "-ar", "24000", "-ac", "1", str(wav_path)],
            check=True,
        )
        mp3_path.unlink()
        info = sf.info(str(wav_path))
        demo_rows.append({
            "id": clip["id"],
            "text": clip["text"],
            "lang": "en-US",
            "audio": wav_path.name,
            "duration_s": round(info.frames / info.samplerate, 3),
            "sample_rate": info.samplerate,
            "source": {"kind": "tts", "plugin": "ovos-tts-plugin-edge-tts", "voice": DEMO_VOICE, "version": "0.3.2a2"},
            "identity": "miro",
            "scores": {"utmos": clip["utmos"], "cer": clip["cer"]},
        })

    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in demo_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(str(manifest_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump({
            "notebooks": ["tts_dataset_gen (demo)"],
            "instruments": {
                "utmos": {"package": "speechonnxmetrics", "version": "0.0.2a2", "model": "utmos-v2"},
                "cer": {"package": "ovos-stt-plugin-rover", "version": "0.0.1a1", "model": "rover-consensus"},
            },
        }, f, indent=2)

    kept_rows = [r for r in demo_rows if r["scores"]["cer"] <= 0.10]
    removed = len(demo_rows) - len(kept_rows)
    selection_path = src_dir / "selection.jsonl"
    with open(selection_path, "w", encoding="utf-8") as f:
        for row in kept_rows:
            f.write(json.dumps(row["id"]) + "\n")
    with open(str(selection_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump({"rules": [{"rule": "cer <= 0.10", "removed": removed}]}, f, indent=2)

    if not _env("SELECTION_PATH"):
        os.environ["SELECTION_PATH"] = str(selection_path)

    print(f"Demo manifest: {len(demo_rows)} rows, {len(kept_rows)} pass cer <= 0.10, {removed} fails it.")
else:
    print("Using the given manifest and selection as-is.")


Demo manifest: 4 rows, 3 pass cer <= 0.10, 1 fails it.


---
## 2 · Build the dataset folder

Reads the manifest and the selection, then writes `wavs/`, `metadata.csv` and
the full `manifest.jsonl`. Nothing is deleted or moved from the manifest's own
audio: files are hardlinked into `wavs/` when they share a filesystem with
`OUTPUT_DIR`, copied otherwise.

A row that a step made by voice conversion or enhancement carries no `source`
of its own (see `docs/manifest.md`); this stage walks `parent_id` back to the
ancestor row that does, so the card below can report where the speech
originally came from.

| Env var | Meaning |
|---|---|
| `OUTPUT_DIR` | The Hugging Face dataset folder to build. |


In [4]:
import json
import os
import shutil
from pathlib import Path


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_selection(path):
    """A selection file is a JSON Lines file of kept id values (docs/manifest.md).
    Returns None when no selection file is given: every row is then selected."""
    if not path or not os.path.exists(path):
        return None
    kept = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if isinstance(obj, str):
                kept.add(obj)
            elif isinstance(obj, dict) and "id" in obj:
                kept.add(obj["id"])
            else:
                raise ValueError(f"Selection line is neither an id string nor an object with 'id': {line!r}")
    return kept


def load_sidecar(manifest_or_selection_path):
    """The .meta.json sidecar next to a manifest or selection file. An absent
    sidecar is reported as absent, not fabricated: returns {}."""
    sidecar_path = manifest_or_selection_path + ".meta.json"
    if os.path.exists(sidecar_path):
        with open(sidecar_path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def resolve_source(row, by_id):
    """Walk parent_id back to the ancestor row that carries a 'source'."""
    seen = set()
    r = row
    while r is not None:
        if "source" in r:
            return r["source"]
        pid = r.get("parent_id")
        if pid is None or pid in seen:
            return None
        seen.add(pid)
        r = by_id.get(pid)
    return None


def source_label(source):
    if source is None:
        return "unknown"
    if source.get("kind") == "tts":
        return f"donor TTS: {source.get('plugin')} / {source.get('voice')}"
    if source.get("kind") == "corpus":
        name = source.get("name", "unknown corpus")
        url = source.get("url")
        return f"{name} ({url})" if url else name
    return "unknown"


def copy_or_link(src, dst):
    """Hardlink when src and dst share a filesystem; copy otherwise."""
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def selected_ids(rows, selection):
    if selection is None:
        return {r["id"] for r in rows}
    return {r["id"] for r in rows if r["id"] in selection}


def build_dataset_folder(manifest_path, selection_path, output_dir):
    rows = load_jsonl(manifest_path)
    by_id = {r["id"]: r for r in rows}
    selection = load_selection(selection_path)
    kept = selected_ids(rows, selection)

    manifest_dir = Path(manifest_path).parent
    wavs_dir = Path(output_dir) / "wavs"
    if wavs_dir.exists():
        # A rebuild starts clean: an old wavs/ can hold a hardlink to audio a
        # fresh demo run just deleted and recreated at the same path.
        shutil.rmtree(wavs_dir)
    wavs_dir.mkdir(parents=True)

    meta_lines = []
    for row in sorted(rows, key=lambda r: r["id"]):
        if row["id"] not in kept:
            continue
        src = manifest_dir / row["audio"]
        dst = wavs_dir / f"{row['id']}.wav"
        copy_or_link(src, dst)
        meta_lines.append(f"{row['id']}|{row['text']}")

    with open(Path(output_dir) / "metadata.csv", "w", encoding="utf-8") as f:
        f.write("\n".join(meta_lines) + ("\n" if meta_lines else ""))

    out_manifest_path = Path(output_dir) / "manifest.jsonl"
    with open(out_manifest_path, "w", encoding="utf-8") as f:
        for row in rows:
            out_row = dict(row)
            out_row["selected"] = row["id"] in kept
            f.write(json.dumps(out_row, ensure_ascii=False) + "\n")

    manifest_sidecar = load_sidecar(manifest_path)
    selection_sidecar = load_sidecar(selection_path) if selection_path else {}
    out_sidecar = {
        "notebooks": manifest_sidecar.get("notebooks", []) + ["tts_publish"],
        "instruments": manifest_sidecar.get("instruments", {}),
        "selection": (
            {"path": os.path.basename(selection_path), **selection_sidecar}
            if selection_path and selection is not None
            else {"applied": False, "note": "no selection file was given; every row is selected"}
        ),
    }
    with open(str(out_manifest_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump(out_sidecar, f, ensure_ascii=False, indent=2)

    return rows, by_id, kept, out_manifest_path


PUBLISH_ROWS, PUBLISH_BY_ID, PUBLISH_KEPT, PUBLISH_MANIFEST_OUT = build_dataset_folder(
    _env("MANIFEST_PATH"), _env("SELECTION_PATH"), _env("OUTPUT_DIR")
)
print(f"{len(PUBLISH_ROWS)} rows in the manifest, {len(PUBLISH_KEPT)} selected and copied into wavs/.")


4 rows in the manifest, 3 selected and copied into wavs/.


---
## 3 · Generate the dataset card

The card is built from the selected rows and the sidecars, never typed by
hand: language and variety, voice identity, source, clip count and hours,
sample rate, the scores present with their instrument and version, and the
selection rules applied. The ownership and licence block matches the rest of
the collection.


In [5]:
OWNERSHIP_BLOCK = """## Ownership and licensing

Miro and Dii are the recorded voices of two real people. The voices, this
dataset, and any model trained on it belong to TigreGotico Lda
(https://tigregotico.pt).

This dataset is licensed under Creative Commons
Attribution-NonCommercial-NoDerivatives 4.0 (CC BY-NC-ND 4.0). You may use
and share this dataset for non-commercial purposes only. Give attribution to
TigreGotico Lda. Do not modify, remix, or build derivative datasets or voices
from this data.

For commercial use, for derivative datasets, or for any other license of the
Miro or Dii voice identity, contact TigreGotico Lda."""


def build_card(repo_name, rows, by_id, kept, sidecar):
    selected_rows = [r for r in rows if r["id"] in kept]

    langs = sorted({r.get("lang", "unknown") for r in selected_rows})
    identities = sorted({r.get("identity") for r in selected_rows if r.get("identity")})
    if len(identities) == 1:
        identity = identities[0]
    elif identities:
        identity = "mixed: " + ", ".join(identities)
    else:
        identity = "none recorded (no voice conversion in this manifest)"

    srs = sorted({r.get("sample_rate") for r in selected_rows if r.get("sample_rate")})
    sample_rate = str(srs[0]) if len(srs) == 1 else (", ".join(str(s) for s in srs) if srs else "not recorded")

    total_hours = sum(r.get("duration_s", 0) for r in selected_rows) / 3600.0

    sources = {}
    for row in selected_rows:
        label = source_label(resolve_source(row, by_id))
        sources[label] = sources.get(label, 0) + 1

    score_keys = sorted({k for r in selected_rows for k in r.get("scores", {}).keys()})
    instruments = sidecar.get("instruments", {})
    selection_info = sidecar.get("selection", {})

    lines = []
    lines.append("---")
    lines.append("language:")
    for lg in langs:
        lines.append(f"- {lg.split('-')[0]}")
    lines.append("license: cc-by-nc-nd-4.0")
    lines.append("task_categories:")
    lines.append("- text-to-speech")
    lines.append("tags:")
    lines.append("- tts")
    lines.append("- speech")
    lines.append("---")
    lines.append("")
    lines.append(f"# {repo_name}")
    lines.append("")
    lines.append(
        f"This is a single-speaker synthetic speech dataset for **{', '.join(langs)}**. "
        f"It carries the **{identity}** voice identity. "
        f"The dataset contains {len(selected_rows)} recordings at {sample_rate} Hz, "
        f"{total_hours:.4f} hours of audio, with a `metadata.csv` transcript file "
        f"and one WAV file per line."
    )
    lines.append("")
    lines.append("## Source")
    lines.append("")
    for label, count in sorted(sources.items()):
        lines.append(f"- {label}: {count} clips")
    lines.append("")
    lines.append("## Scores")
    lines.append("")
    if score_keys:
        for key in score_keys:
            instr = instruments.get(key)
            if instr:
                model = f", model {instr['model']}" if instr.get("model") else ""
                lines.append(f"- `{key}`: {instr.get('package', 'unknown package')} {instr.get('version', 'unknown version')}{model}")
            else:
                lines.append(f"- `{key}`: instrument not recorded in the manifest sidecar")
    else:
        lines.append("No scores are recorded on the selected rows.")
    lines.append("")
    lines.append("## Selection")
    lines.append("")
    if selection_info.get("applied") is False:
        lines.append(selection_info.get("note", "No selection file was given; every row is selected."))
    else:
        rules = selection_info.get("rules", [])
        if rules:
            for rule in rules:
                lines.append(f"- `{rule.get('rule')}` removed {rule.get('removed')} rows")
        else:
            lines.append("A selection file was applied; its sidecar records no rules.")
    lines.append("")
    lines.append(OWNERSHIP_BLOCK)
    lines.append("")
    return "\n".join(lines)


PUBLISH_SIDECAR = load_sidecar(str(PUBLISH_MANIFEST_OUT))
repo_name = _env("HF_REPO").split("/")[-1] or Path(_env("OUTPUT_DIR")).name
PUBLISH_CARD = build_card(repo_name, PUBLISH_ROWS, PUBLISH_BY_ID, PUBLISH_KEPT, PUBLISH_SIDECAR)
(Path(_env("OUTPUT_DIR")) / "README.md").write_text(PUBLISH_CARD, encoding="utf-8")
print(PUBLISH_CARD)


---
language:
- en
license: cc-by-nc-nd-4.0
task_categories:
- text-to-speech
tags:
- tts
- speech
---

# dataset

This is a single-speaker synthetic speech dataset for **en-US**. It carries the **miro** voice identity. The dataset contains 3 recordings at 24000 Hz, 0.0026 hours of audio, with a `metadata.csv` transcript file and one WAV file per line.

## Source

- donor TTS: ovos-tts-plugin-edge-tts / en-US-JennyNeural: 3 clips

## Scores

- `cer`: ovos-stt-plugin-rover 0.0.1a1, model rover-consensus
- `utmos`: speechonnxmetrics 0.0.2a2, model utmos-v2

## Selection

- `cer <= 0.10` removed 1 rows

## Ownership and licensing

Miro and Dii are the recorded voices of two real people. The voices, this
dataset, and any model trained on it belong to TigreGotico Lda
(https://tigregotico.pt).

This dataset is licensed under Creative Commons
Attribution-NonCommercial-NoDerivatives 4.0 (CC BY-NC-ND 4.0). You may use
and share this dataset for non-commercial purposes only. Give attribution to


---
## 4 · Push to Hugging Face (optional)

Off by default. With `HF_PUSH=1`, the folder is pushed into `HF_REPO` (for
example `TigreGotico/tts-train-synthetic-miro_pt-PT`), one commit through
`upload_folder` for a folder of up to 10,000 files, `upload_large_folder`
above that. After the push, the repo's file list is read back and the
pushed WAV count is checked against the number of selected rows.

| Env var | Meaning |
|---|---|
| `HF_PUSH` | `1` to push `OUTPUT_DIR` to the Hub. |
| `HF_REPO` | Target dataset repo, for example `TigreGotico/tts-train-synthetic-miro_pt-PT`. |
| `HF_TOKEN` | Write token (or use `huggingface-cli login`). |


In [6]:
if _flag("HF_PUSH"):
    !pip install --quiet --break-system-packages --pre "huggingface_hub>=2.0.0"
else:
    print("HF_PUSH not enabled — skipping install.")


HF_PUSH not enabled — skipping install.


In [7]:
if not _flag("HF_PUSH"):
    n_wavs = len(list((Path(_env("OUTPUT_DIR")) / "wavs").glob("*.wav")))
    print("HF_PUSH=0 — not pushing. Would upload:")
    print(f"  repo       : {_env('HF_REPO') or '(HF_REPO not set)'}")
    print(f"  folder     : {_env('OUTPUT_DIR')}")
    print(f"  wavs       : {n_wavs}")
    print(f"  metadata   : metadata.csv, manifest.jsonl, manifest.jsonl.meta.json, README.md")
else:
    repo_id = _env("HF_REPO")
    if not repo_id:
        print("ERROR: HF_REPO not set. Example: TigreGotico/tts-train-synthetic-miro_pt-PT")
    else:
        from huggingface_hub import HfApi, upload_large_folder

        api = HfApi(token=_env("HF_TOKEN") or None)
        api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

        output_dir = Path(_env("OUTPUT_DIR"))
        n_files = sum(1 for p in output_dir.rglob("*") if p.is_file())
        print(f"Pushing {n_files} files to {repo_id} ...")

        if n_files > 10000:
            # upload_large_folder batches the commit itself.
            upload_large_folder(repo_id=repo_id, repo_type="dataset", folder_path=str(output_dir))
        else:
            api.upload_folder(
                folder_path=str(output_dir),
                repo_id=repo_id,
                repo_type="dataset",
                commit_message="Publish TTS dataset",
            )

        remote_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
        remote_wavs = [f for f in remote_files if f.startswith("wavs/") and f.endswith(".wav")]
        expected = len(PUBLISH_KEPT)
        assert len(remote_wavs) == expected, (
            f"pushed {len(remote_wavs)} wavs, expected {expected} selected rows"
        )
        print(f"Verified: {len(remote_wavs)} wavs on the Hub match {expected} selected rows.")
        print(f"  https://huggingface.co/datasets/{repo_id}")


HF_PUSH=0 — not pushing. Would upload:
  repo       : (HF_REPO not set)
  folder     : ./tts_publish_demo/dataset
  wavs       : 3
  metadata   : metadata.csv, manifest.jsonl, manifest.jsonl.meta.json, README.md


---
## 5 · Output summary


In [8]:
output_dir = Path(_env("OUTPUT_DIR"))
n_wavs = len(list((output_dir / "wavs").glob("*.wav")))
meta_n = sum(1 for line in open(output_dir / "metadata.csv", encoding="utf-8") if line.strip())
manifest_n = sum(1 for line in open(output_dir / "manifest.jsonl", encoding="utf-8") if line.strip())

print("=" * 60)
print("  OUTPUT SUMMARY")
print("=" * 60)
print(f"  {'wavs/':20s} {n_wavs:>6} files")
print(f"  {'metadata.csv':20s} {meta_n:>6} rows")
print(f"  {'manifest.jsonl':20s} {manifest_n:>6} rows ({n_wavs} selected, {manifest_n - n_wavs} not)")
marker = "OK" if n_wavs == meta_n else "MISMATCH"
print(f"\n  wavs vs metadata.csv: {marker}")


  OUTPUT SUMMARY
  wavs/                     3 files
  metadata.csv              3 rows
  manifest.jsonl            4 rows (3 selected, 1 not)

  wavs vs metadata.csv: OK
